<a href="https://colab.research.google.com/github/divyasri2609/Medical-Transcriptions/blob/main/RAG_SYSTEM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas numpy sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.3 MB/s eta 0:00:00


In [3]:
import pandas as pd

from google.colab import files
uploaded = files.upload()

import zipfile
import os

zip_path = "/content/archive (10).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")

print("Files extracted:", os.listdir("data"))

Saving archive (10).zip to archive (10).zip
Files extracted: ['mtsamples.csv']


In [6]:
import re
import pandas as pd

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df = pd.read_csv('data/mtsamples.csv')
df['clean_text'] = df['transcription'].fillna('').apply(clean_text)

In [11]:
top_classes = df['medical_specialty'].value_counts().head(10).index
df = df[df['medical_specialty'].isin(top_classes)]

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['clean_text'].tolist()

embeddings = model.encode(texts, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/115 [00:00<?, ?it/s]

In [12]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors:", index.ntotal)

Total vectors: 3657


In [13]:
def retrieve(query, k=5):
    query_clean = clean_text(query)

    query_vec = model.encode([query_clean])

    distances, indices = index.search(query_vec, k)

    return df.iloc[indices[0]]

In [14]:
query = "patient has chest pain and shortness of breath"

results = retrieve(query)

results[['medical_specialty', 'transcription']].head()

,medical_specialty,transcription
4096,Consult - History and Phy.,"REASON FOR CONSULTATION:, Ventricular ectopy ..."
4609,Cardiovascular / Pulmonary,"REASON FOR CONSULTATION:, Ventricular ectopy ..."
4149,Consult - History and Phy.,"HISTORY: ,This 61-year-old retailer who prese..."
4684,Cardiovascular / Pulmonary,"HISTORY: ,This 61-year-old retailer who prese..."
4592,Consult - History and Phy.,"REASON FOR CONSULTATION: ,Abnormal echocardio..."


In [15]:
def generate_response(query, retrieved_df):

    specialty = retrieved_df['medical_specialty'].value_counts().idxmax()

    return f"""
    🔹 Predicted Specialty: {specialty}

    🔹 Clinical Insight:
    Based on similar retrieved cases, the symptoms may indicate a condition related to {specialty}.

    🔹 Evidence:
    Top {len(retrieved_df)} similar cases were analyzed.
    """

In [16]:
print(generate_response(query, results))


    🔹 Predicted Specialty:  Consult - History and Phy.
    
    🔹 Clinical Insight:
    Based on similar retrieved cases, the symptoms may indicate a condition related to  Consult - History and Phy..
    
    🔹 Evidence:
    Top 5 similar cases were analyzed.
    


In [17]:
queries = [
    "chest pain and heart issues",
    "brain tumor symptoms",
    "bone fracture injury",
    "skin infection rash",
    "kidney failure symptoms"
]

for q in queries:
    print("\n====================")
    print("Query:", q)

    res = retrieve(q)

    print("Top Matches:")
    print(res['medical_specialty'].head(3).tolist())


Query: chest pain and heart issues
Top Matches:
[' Cardiovascular / Pulmonary', ' Radiology', ' Cardiovascular / Pulmonary']

Query: brain tumor symptoms
Top Matches:
[' SOAP / Chart / Progress Notes', ' Neurology', ' Neurology']

Query: bone fracture injury
Top Matches:
[' Surgery', ' Orthopedic', ' Surgery']

Query: skin infection rash
Top Matches:
[' Consult - History and Phy.', ' Consult - History and Phy.', ' General Medicine']

Query: kidney failure symptoms
Top Matches:
[' Consult - History and Phy.', ' SOAP / Chart / Progress Notes', ' General Medicine']


In [18]:
model = SentenceTransformer('all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
df = df[df['clean_text'].str.len() > 50]

In [21]:
def weighted_specialty(retrieved_df):
    return retrieved_df['medical_specialty'].mode()[0]